# 3. Команды и успех
Первый взгляд: связан ли размер команды с итоговым местом.
Берём только Featured-соревнования — там самая серьёзная конкуренция.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load

sns.set_theme(style="whitegrid")
comp = load("Competitions")
teams = load("Teams")
members = load("TeamMemberships")

In [ ]:
featured = comp.loc[comp["HostSegmentTitle"] == "Featured", ["Id", "TotalTeams"]]
teams = teams.merge(featured, left_on="CompetitionId", right_on="Id", suffixes=("", "_comp"))
teams = teams.dropna(subset=["PrivateLeaderboardRank"])
len(teams)

Размер команды = число участников в TeamMemberships.
Место нормируем на число команд: 0 — первое место, 1 — последнее.

In [ ]:
size = members.groupby("TeamId").size().rename("team_size")
teams = teams.merge(size, left_on="Id", right_index=True, how="left")
teams["team_size"] = teams["team_size"].fillna(1).clip(upper=5)  # 5 = "5 и больше"
teams["rank_pct"] = teams["PrivateLeaderboardRank"] / teams["TotalTeams"]
teams["team_size"].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=teams, x="team_size", y="rank_pct")
plt.title("Относительное место в зависимости от размера команды (5 = 5+)")
plt.ylabel("место / число команд")
plt.savefig("../reports/figures/03_rank_by_team_size.png", bbox_inches="tight")
plt.show()

In [ ]:
teams["has_medal"] = teams["Medal"].notna()
teams.groupby("team_size")["has_medal"].mean()

TODO: добавить опыт участников (уровень, число прошлых соревнований) и число посылок из Submissions.